In [1]:
# Import dependencies
import numpy as np
import matplotlib.pyplot as plt
from src.evaluate import evaluate
from src.data.preprocessing import create_data_loaders
from src.utils.visualization import visualize_predictions

# Initialize
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)

ModuleNotFoundError: No module named 'src'

In [ ]:
# Run evaluation with optimal threshold (0.4 based on previous results)
results = evaluate(threshold=0.4)

In [ ]:
# Get indices of errors
fp_indices = np.where((results['labels'] == 0) & (results['preds'] == 1))[0]  # False Pneumonia
fn_indices = np.where((results['labels'] == 1) & (results['preds'] == 0))[0]  # Missed Pneumonia

print(f"False Positives (Normal predicted as Pneumonia): {len(fp_indices)}")
print(f"False Negatives (Pneumonia predicted as Normal): {len(fn_indices)}")

In [ ]:
# Load test data
_, _, test_loader = create_data_loaders('data/raw/chest_xray')
test_dataset = test_loader.dataset

def show_errors(indices, title):
    """Display images with true/predicted labels"""
    plt.figure(figsize=(15, 5))
    for i, idx in enumerate(indices[:8]):  # Show first 8 errors
        image, label = test_dataset[idx]
        prob = results['probs'][idx]
        
        plt.subplot(2, 4, i+1)
        plt.imshow(image.permute(1, 2, 0).numpy())
        plt.title(f"True: {'PNEUMONIA' if label else 'NORMAL'}\n"
                  f"Pred: {'PNEUMONIA' if results['preds'][idx] else 'NORMAL'}\n"
                  f"Conf: {max(prob):.2f}")
        plt.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# Display errors
if len(fp_indices) > 0:
    show_errors(fp_indices, "False Positives (Normal X-rays predicted as Pneumonia)")
    
if len(fn_indices) > 0:
    show_errors(fn_indices, "False Negatives (Pneumonia X-rays predicted as Normal)")

In [ ]:
# Plot confidence distributions
plt.figure(figsize=(12, 5))

# Correct predictions
correct_mask = (results['labels'] == results['preds'])
plt.hist(np.max(results['probs'][correct_mask], bins=20, 
         alpha=0.7, label='Correct', color='green')

# Incorrect predictions
incorrect_mask = (results['labels'] != results['preds'])
plt.hist(np.max(results['probs'][incorrect_mask], bins=20, 
         alpha=0.7, label='Incorrect', color='red')

plt.xlabel('Model Confidence')
plt.ylabel('Count')
plt.title('Prediction Confidence Distribution')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Test multiple thresholds
thresholds = np.linspace(0.1, 0.9, 9)
metrics = []

for thresh in thresholds:
    preds = (results['probs'][:, 1] > thresh).astype(int)
    correct = (preds == results['labels']).mean()
    metrics.append({
        'threshold': thresh,
        'accuracy': correct,
        'false_positives': ((preds == 1) & (results['labels'] == 0)).sum(),
        'false_negatives': ((preds == 0) & (results['labels'] == 1)).sum()
    })

# Plot results
plt.figure(figsize=(12, 5))
plt.plot(thresholds, [m['accuracy'] for m in metrics], label='Accuracy')
plt.plot(thresholds, [m['false_positives'] for m in metrics], label='False Positives')
plt.plot(thresholds, [m['false_negatives'] for m in metrics], label='False Negatives')
plt.xlabel('Threshold')
plt.ylabel('Count/Accuracy')
plt.title('Threshold Optimization')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Visualize random predictions
visualize_predictions(
    model=model,
    dataloader=test_loader,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    num_images=8
)